# 📚 Fiche de révision — Théorie Bayésienne

Ce notebook reprend tous les concepts théoriques du cours, exercice par exercice.
Pas de code ici, que de la théorie expliquée simplement.

---


## PARTIE 1 — Fréquence, estimateur et variabilité

### C'est quoi une fréquence empirique ?

Quand on observe des données, la **fréquence empirique** c'est juste le ratio de fois où un événement s'est produit. Par exemple, si on a 150 observations et 18 anomalies, la fréquence c'est 18/150 = 0.12.

C'est la chose la plus naturelle à calculer, mais ça n'est pas parfait.

---

### Est-ce que c'est un bon estimateur ?

Oui, au sens statistique, c'est un bon estimateur pour deux raisons :

**1. Sans biais** : si on refaisait l'expérience des millions de fois et qu'on moyennait tous les résultats, on tomberait exactement sur la vraie valeur p. En gros, en moyenne on ne se trompe pas.

**2. Consistant** : plus on a de données, plus on s'approche de la vraie valeur. Avec n=50 on peut être loin, avec n=10000 on sera très proche.

> ⚠️ Attention : ça ne veut pas dire que sur *un seul* échantillon on tombe juste. Ça veut juste dire qu'on ne fait pas d'erreur systématique.

---

### Pourquoi deux expériences identiques donnent des résultats différents ?

Parce que chaque tirage est aléatoire. On simule `np.random.binomial(1, 0.12, 150)` : chaque observation est un pile ou face avec une proba de 0.12 de tomber sur 1. Deux runs différents vont forcément donner des séquences différentes.

C'est pas une erreur, c'est juste la nature du hasard. C'est pour ça qu'on fixe une **seed** (`np.random.seed(123)`) : pour que tout le monde ait exactement les mêmes résultats.

---

### Qu'est-ce qu'on voit quand on change n ?

| n | Écart-type des fréquences | Ce que ça veut dire |
|---|---|---|
| 50 | ≈ 0.047 | On peut tomber entre 0.05 et 0.19, c'est large |
| 150 | ≈ 0.026 | Un peu mieux |
| 800 | ≈ 0.011 | Très concentré autour de 0.12 |

Plus n est grand, plus la distribution des fréquences est **serrée autour de la vraie valeur**. C'est la **loi des grands nombres**.

---


## PARTIE 2 — Hypothèses discrètes et variable latente

### Le contexte

On a un système de sécurité qui génère des alertes. On ne sait pas si le système est normal (H=0) ou compromis (H=1). On cherche à deviner lequel des deux à partir des alertes observées.

---

### C'est quoi une variable latente ?

Une **variable latente** c'est une variable qui existe mais qu'on ne peut pas observer directement. Ici, H (l'état du système) est latente : on ne voit que les alertes, pas si le système est vraiment compromis ou pas.

Sans H, on ne peut pas expliquer d'où viennent les alertes. Est-ce un faux positif du système normal ? Ou une vraie détection du système compromis ? On ne peut pas savoir sans modéliser H explicitement.

---

### La structure causale du modèle

```
H (état du système)
        │
        ▼
   taux d'alertes    ←── (différent selon H=0 ou H=1)
        │
        ▼
 alertes observées
```

La **cause** c'est H. L'**effet** c'est ce qu'on observe (les alertes). On essaie de remonter de l'effet vers la cause : c'est exactement ce que fait Bayes.

---

### Comment calculer P(H=1 | données) ?

On utilise le **théorème de Bayes** :

```
P(H=1 | données) ∝ P(données | H=1) × P(H=1)
```

- **P(H=1)** : le prior, notre croyance avant de voir les données. Si on n'a pas d'info, on met 0.5 (50/50).
- **P(données | H=1)** : la vraisemblance, c'est-à-dire "si le système était compromis, quelle est la proba d'observer autant d'alertes ?"
- **P(H=1 | données)** : le posterior, notre croyance mise à jour après avoir vu les données.

---

### Pourquoi le résultat est proche de 0.5 dans cet exercice ?

Parce que les deux hypothèses génèrent des taux d'alertes très proches :
- H=0 (normal) : P(alerte) ≈ 0.215
- H=1 (compromis) : P(alerte) ≈ 0.190

Les données ne permettent pas vraiment de distinguer les deux. Avec un prior 50/50, on reste dans le doute. Ce n'est pas un bug, c'est une conclusion honnête : le signal est trop faible.

---


## PARTIE 3 — Inférence bayésienne et décision

### Le modèle Beta-Binomial

C'est LE modèle bayésien de base pour estimer une proportion.

**Situation** : on observe k succès sur n essais, et on veut estimer la vraie probabilité p de succès.

**Modèle** :
- Prior : `p ~ Beta(α, β)` — notre croyance sur p avant les données
- Vraisemblance : `k | p ~ Binomial(n, p)` — les données
- Posterior : `p | k ~ Beta(α + k, β + n - k)` — notre croyance mise à jour

C'est la **conjugaison** : le prior Beta + la vraisemblance Binomiale = posterior Beta. C'est pratique parce qu'on n'a pas besoin de calculer numériquement, on a la formule directement.

---

### C'est quoi un prior non-informatif ?

Un **prior non-informatif** (ou non-informatif) c'est `Beta(1, 1)` = distribution uniforme sur [0,1]. Ça dit qu'on ne favorise aucune valeur de p, toutes sont également plausibles avant de voir les données.

Avec 13 succès sur 20 et un prior Beta(1,1) :
```
Posterior = Beta(1 + 13, 1 + 7) = Beta(14, 8)
Moyenne a posteriori = 14 / (14 + 8) = 14/22 ≈ 0.636
```

---

### C'est quoi un prior informatif ?

Un **prior informatif** comme `Beta(6, 2)` dit qu'on pense déjà que p est plutôt élevé (autour de 0.75). Ce prior vient d'une connaissance préalable (expériences passées, expertise métier...).

Avec le même jeu de données :
```
Posterior = Beta(6 + 13, 2 + 7) = Beta(19, 9)
Moyenne a posteriori = 19/28 ≈ 0.679
```

Le prior optimiste "tire" le résultat vers le haut.

---

### Pourquoi le prior a plus d'impact avec peu de données ?

C'est une conséquence directe de la formule :
```
Posterior = Beta(α + k, β + n - k)
```

Si n = 20 : α et β du prior restent importants relativement à k et n-k.
Si n = 2000 : k et n-k écrasent α et β, le prior devient négligeable.

**Intuition** : si on n'a que 5 observations, notre a priori compte autant que les données. Si on en a 10 000, les données parlent d'elles-mêmes et le prior ne change quasi rien.

---

### Comment prendre une décision ?

On utilise le posterior pour calculer des probabilités :
- `P(p > 0.75)` = proba que le vrai taux dépasse le seuil de déploiement
- Si cette proba est assez grande → on déploie
- Sinon → on attend plus de données ou on abandonne

| Règle | Décision |
|---|---|
| Moyenne posterior ≥ 0.75 | Déploiement |
| 0.50 – 0.75 | Test supplémentaire |
| < 0.50 | Abandon |

---


## PARTIE 4 — Comparaison de modèles et validation

### Pourquoi comparer des modèles ?

En bayésien, quand on a plusieurs modèles possibles (M1, M2...), on veut savoir lequel explique le mieux les données. On ne choisit pas "à la main", on laisse les données parler.

---

### Le Bayes Factor

Le **Facteur de Bayes** (BF) mesure combien les données favorisent un modèle par rapport à un autre :

```
BF₂₁ = P(données | M2) / P(données | M1)
```

- BF > 1 → M2 explique mieux les données
- BF < 1 → M1 explique mieux les données

**Échelle d'interprétation** :

| BF | Force de l'évidence |
|---|---|
| 1 – 3 | Faible |
| 3 – 10 | Modérée |
| 10 – 30 | Forte |
| > 100 | Décisive |

En pratique dans PyMC on utilise **LOO** (Leave-One-Out) ou **WAIC** qui sont des approximations computationnelles du Bayes Factor. Un LOO plus élevé = meilleur modèle.

---

### Un bon score de comparaison garantit-il un bon modèle ?

**Non.** C'est le piège classique.

Le LOO/WAIC mesure seulement lequel des deux modèles comparés est *le moins mauvais*. Mais si les deux modèles sont mauvais, le gagnant est quand même mauvais.

Exemple : si M1 prédit des valeurs entre 6 et 8, et M2 prédit entre 5 et 10, M2 "gagne" face à M1. Mais si les vraies données sont toutes soit autour de 5 soit autour de 10, M2 est quand même mauvais.

---

### Le Posterior Predictive Check (PPC)

C'est la validation du modèle. L'idée c'est simple :

> Si mon modèle est bon, il doit être capable de générer des données qui ressemblent aux vraies données.

**Comment ça marche :**
1. On estime les paramètres du modèle avec les données réelles
2. On utilise ces paramètres pour simuler de *nouvelles* données (appelées y_rep)
3. On compare ces données simulées aux données réelles

Si les simulations ressemblent aux vraies données → modèle crédible.
Si les simulations sont complètement à côté → modèle mal spécifié, à revoir.

**Résumé** :
- Bayes Factor / LOO → répond à "lequel est meilleur ?"
- PPC → répond à "est-ce que le modèle est réaliste ?"
- Les deux sont complémentaires, il faut faire les deux.

---

### C'est quoi "deux régimes" dans les données ?

Dans l'exercice 4, les données sont `[5.1, 5.3, 5.0, 5.2, 9.8, 10.1, 9.9, 10.2]`.

On voit clairement deux groupes distincts, très éloignés l'un de l'autre. Une seule loi normale avec une moyenne unique (≈7.7) ne peut pas décrire ça : elle produirait des valeurs autour de 7-8 qui n'existent pas dans les données.

Un modèle à deux régimes (deux moyennes séparées) capture exactement cette structure bimodale. C'est pour ça que M2 gagne largement face à M1.

---


## PARTIE 5 — Récap des concepts clés à connaître

### Le théorème de Bayes (la formule centrale)

```
P(paramètre | données) ∝ P(données | paramètre) × P(paramètre)
       posterior              vraisemblance           prior
```

- **Prior** : ce qu'on croit avant de voir les données
- **Vraisemblance** : à quel point les données sont compatibles avec chaque valeur du paramètre
- **Posterior** : notre croyance mise à jour après les données

---

### La loi Beta — à connaître par cœur

La loi `Beta(α, β)` modélise une probabilité inconnue p ∈ [0,1].

| Paramètre | Intuition |
|---|---|
| α | Nombre de succès "virtuels" (ou réels + prior) |
| β | Nombre d'échecs "virtuels" (ou réels + prior) |
| Moyenne | α / (α + β) |
| Plus α+β est grand | Plus on est confiant (distribution étroite) |

Cas spéciaux :
- `Beta(1,1)` = prior uniforme, on ne sait rien
- `Beta(6,2)` = on pense que p est plutôt élevé (optimiste)
- `Beta(2,6)` = on pense que p est plutôt bas (pessimiste)

---

### Le MAP vs la Moyenne a posteriori

- **MAP** (Maximum A Posteriori) : la valeur de p qui maximise le posterior
  - Formule pour Beta : `MAP = (α-1) / (α+β-2)`
- **Moyenne** : `E[p] = α / (α+β)`

Les deux sont proches mais pas identiques. La moyenne est généralement préférée car elle tient compte de toute la distribution.

---

### Variable latente — quand en a-t-on besoin ?

On a besoin d'une variable latente quand :
- Une cause importante n'est pas directement observable
- Les données peuvent venir de plusieurs mécanismes différents (mélange)
- On cherche à expliquer *pourquoi* on observe quelque chose, pas juste *quoi*

Exemples dans le cours :
- H (état du système) dans l'exercice cybersécurité
- Le groupe (normal vs lumière forte) dans l'exercice plantes

---

### Lissage bayésien — l'idée en une phrase

Quand on a peu de données sur quelque chose, on ne fait pas confiance à la moyenne locale. On la tire vers la moyenne globale. Plus on a de données, moins on corrige.

Formule :
```
score_lissé = (n × moyenne_locale + α × moyenne_globale) / (n + α)
```

C'est exactement le même principe que le posterior bayésien : peu de données → le prior (moyenne globale) domine.

---


## 🎯 Aide-mémoire pour l'exam

### Ce qu'on te demande souvent de "justifier"

**"Justifier le choix du prior"**
→ Prior non-informatif = Beta(1,1) si on n'a aucune info préalable. Prior informatif si on a des données historiques ou une expertise métier. Toujours expliquer pourquoi.

**"Expliquer la structure causale"**
→ Dessiner la chaîne : quelle variable cause quoi. Identifier ce qui est observé vs ce qui est latent.

**"Interpréter le posterior"**
→ Donner la moyenne, l'écart-type, et des probabilités clés (P(p > seuil)). Dire ce que ça implique pour la décision.

**"Expliquer pourquoi deux modèles"**
→ Regarder les données : est-ce qu'elles forment des groupes distincts ? Une seule distribution peut-elle les couvrir ? Si non → plusieurs régimes.

**"Interpréter le PPC"**
→ Si les données simulées ressemblent aux vraies → modèle OK. Si elles sont à côté → revoir le modèle. Ne pas confondre avec la comparaison de modèles.

---

### Les pièges à éviter

❌ Confondre fréquence et probabilité : une fréquence élevée ne veut pas dire qu'une variable est informative.

❌ Penser qu'un bon LOO = bon modèle : c'est relatif, pas absolu.

❌ Oublier de justifier le prior : le choix du prior doit toujours être expliqué.

❌ Ignorer la variabilité du posterior : regarder juste la moyenne ne suffit pas, il faut regarder l'écart-type et les intervalles.

---
